# HAKE-MER — M2 control: sans attention phrase (pool document + têtes par émotion)

**Backbone:** DistilBERT-base · **Modules:** M1 + M2 without phrase cross-attention (shared doc pool + per-label heads).

**Protocol:** batch 16, LR 5e-5, 4 epochs, seeds 42 / 123 / 456.

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0 --device cuda"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
69679c2


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 3.9 MB/s eta 0:00:00


In [4]:
!./run_m1_m2_controls_campaign.sh --variant no_xattn --backbone distilbert-base-uncased {TRAIN_FLAGS}

config.json: 100% 483/483 [00:00<00:00, 5.09MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 501kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 14.2MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 5.99MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 55.1MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.77M [00:00<?, ?B/s]
simplified/train-00000-of-00001.parquet: downloading bytes:  93% 2.58M/2.77M [00:00<00:00, 4.22MB/s]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:00<00:00, 3.54MB/s,  265kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:00<00:00, 3.59MB/s,  270kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:   0% 0.00/350k [00:00<?, ?B/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:00<00:00, 705kB/s, 34.2kB/s  ]
simplified/validation-00000-of-00001.par(…): reconstructing file: 100% 350k/350k [00:00<00:00, 714kB

In [5]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_m2_no_xattn_distilbert_base_uncased_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

  f1_micro: 0.5799 ± 0.0009
  f1_macro: 0.5037 ± 0.0037
  exact_match: 0.4550 ± 0.0026
  map: 0.5108 ± 0.0021


In [6]:
import zipfile
from google.colab import files

slug = "distilbert_base_uncased"
campaign = Path(f"reference/artifacts/m1_m2_no_xattn_{slug}_campaign.json")
out_name = "m1_m2_no_xattn_distilbert_step0.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(campaign, campaign.name)
    for m in sorted(Path("runs").glob(f"{slug}_seed*_m1_m2_no_xattn/metrics.json")):
        zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name}")
files.download(str(zip_path))

Download m1_m2_no_xattn_distilbert_step0.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Archive

Download executed **.ipynb** + zip and hand off for report integration.